In [1]:
from __future__ import annotations

import os
import sys

import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

sys.path.append(r'C:\Users\Stefa\Documents\studio-paper')


from parfitt_trb import TRBConfig, penetration_vs_actual, pwsd, run_trb
from parfitt_trb import plots
from parfitt_trb.display import rollup_ratio
from parfitt_trb.local_spark import build_local_spark
from examples.synth import simulate_panel

OUT = os.path.join(r'C:\Users\Stefa\Documents\studio-paper\examples', "figuresSte")
os.makedirs(OUT, exist_ok=True)

In [2]:
print(sys.executable)

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

c:\Users\Stefa\Documents\studio-paper\.venv\Scripts\python.exe


In [3]:
PROMO_WEEK = 18
pdf = simulate_panel(
    n_households=6000, weeks=40, K=0.24, a=0.22,
    rbr_start=0.42, rbr_stable=0.24, cat_interval=2, seed=11,
    promo_week=PROMO_WEEK, promo_K=0.12, promo_rbr=0.07
)
pdf["ISOWEEKYEAR"] = pd.to_datetime(pdf["txn_date"]).dt.strftime("%G-%V")

spark = build_local_spark("trb-usage-template")
df = spark.createDataFrame(pdf)            # in production: your cluster DataFrame
cfg = TRBConfig(launch_date="2024-01-01", period_length_days=14,
                buying_index_base="triers")

In [4]:
df.limit(10).show()

+----------+----------+--------------+-----------+------+-----------+
|shopper_id|  txn_date|is_new_product|is_category|volume|ISOWEEKYEAR|
+----------+----------+--------------+-----------+------+-----------+
|        h0|2024-01-03|         false|       true|   1.0|    2024-01|
|        h0|2024-01-17|          true|       true|   1.0|    2024-03|
|        h0|2024-01-31|         false|       true|   1.0|    2024-05|
|        h0|2024-02-14|          true|       true|   1.0|    2024-07|
|        h0|2024-02-28|         false|       true|   1.0|    2024-09|
|        h0|2024-03-13|         false|       true|   1.0|    2024-11|
|        h0|2024-03-27|         false|       true|   1.0|    2024-13|
|        h0|2024-04-10|         false|       true|   1.0|    2024-15|
|        h0|2024-04-24|          true|       true|   1.0|    2024-17|
|        h0|2024-05-08|         false|       true|   1.0|    2024-19|
+----------+----------+--------------+-----------+------+-----------+



In [5]:
cfg = TRBConfig(
    launch_date="2024-01-01"
    # , period_length_days=14
    , bucket_column='ISOWEEKYEAR'
    , buying_index_base="triers"
    , buying_index_window_days = 30
    , rbr_interval_mode="bucket"
    , rbr_bucket_unit="month"         # RBR su intervalli mensili
)

In [6]:
res = run_trb(df, cfg)

In [7]:
pen = res.penetration

In [8]:
print(pen)

print([(res.label(t), p) for t, p in res.penetration.series] )
print(pen.ultimate_penetration)
print(pen.snapshot)


Penetration(denominator='dynamic', origin=datetime.date(2024, 1, 1), series=[(1, 0.04733333333333333), (2, 0.116), (3, 0.16016666666666668), (4, 0.1885), (5, 0.20683333333333334), (6, 0.21866666666666668), (7, 0.22633333333333333), (8, 0.23116666666666666), (9, 0.23433333333333334), (10, 0.31216666666666665), (11, 0.3415), (12, 0.3525), (13, 0.35683333333333334), (14, 0.3585), (15, 0.35933333333333334), (16, 0.3595), (17, 0.35983333333333334), (18, 0.35983333333333334), (19, 0.36)], n_brand_triers=2160, n_category_triers=6000, ultimate_penetration=0.3626197842203338, growth_rate=0.24995892889773308, note='')
[('2024-01', 0.04733333333333333), ('2024-03', 0.116), ('2024-05', 0.16016666666666668), ('2024-07', 0.1885), ('2024-09', 0.20683333333333334), ('2024-11', 0.21866666666666668), ('2024-13', 0.22633333333333333), ('2024-15', 0.23116666666666666), ('2024-17', 0.23433333333333334), ('2024-19', 0.31216666666666665), ('2024-21', 0.3415), ('2024-23', 0.3525), ('2024-25', 0.35683333333333

In [28]:
from parfitt_trb.aggregation import SparkAggregator
from pyspark.sql import functions as F

agg = SparkAggregator(df, cfg)

In [ ]:
p = agg._prepare(df)
p.limit(10).show()
p = p.filter(F.col("ts") <= F.lit(agg._adate).cast("date"))

+----+----------+--------+------+---+-------+
|card|        ts|is_brand|is_cat|qty| bucket|
+----+----------+--------+------+---+-------+
|  h0|2024-01-03|   false|  true|1.0|2024-01|
|  h0|2024-01-17|    true|  true|1.0|2024-03|
|  h0|2024-01-31|   false|  true|1.0|2024-05|
|  h0|2024-02-14|    true|  true|1.0|2024-07|
|  h0|2024-02-28|   false|  true|1.0|2024-09|
|  h0|2024-03-13|   false|  true|1.0|2024-11|
|  h0|2024-03-27|   false|  true|1.0|2024-13|
|  h0|2024-04-10|   false|  true|1.0|2024-15|
|  h0|2024-04-24|    true|  true|1.0|2024-17|
|  h0|2024-05-08|   false|  true|1.0|2024-19|
+----+----------+--------+------+---+-------+



datetime.date(2024, 1, 1)

In [35]:
agg._bucket_to_period

{'2024-01': 1,
 '2024-03': 2,
 '2024-05': 3,
 '2024-07': 4,
 '2024-09': 5,
 '2024-11': 6,
 '2024-13': 7,
 '2024-15': 8,
 '2024-17': 9,
 '2024-19': 10,
 '2024-21': 11,
 '2024-23': 12,
 '2024-25': 13,
 '2024-27': 14,
 '2024-29': 15,
 '2024-31': 16,
 '2024-33': 17,
 '2024-35': 18,
 '2024-37': 19,
 '2024-39': 20}